# Pretrain PairHead + redesigned pairwise critic

This notebook runs the new two-stage pretrain without rebuilding data when the existing GCS caches are present.

Stage 1 trains the full L3/L4 PairHead actor on the existing T=10 pair cache. The actor is built with `PlayerConsolidator` present, but pair supervision does not use it.

Stage 2 trains the redesigned critic on the existing labelled cross cache. The critic objective is `PairCompareHead([player_i || player_j || glob])` at the terminal value horizon plus per-player `is_ahead` BCE derived from existing `leader_seat_t_plus_K` labels. It unfreezes only `PlayerConsolidator`, keeping actor L2 fixed so the actor representation is not shifted.

The final merge cell writes an actor checkpoint with the trained consolidator copied in, so PPO actors can load the merged actor as their base and apply the `pair_compare` head delta.


## 0. Config

In [ ]:
# GCS roots. Existing caches live in separate prefixes; do not copy them between prefixes.
ENTITY_BUCKET = 'gs://orbit-wars-shipping/entity'
CROSS_BUCKET  = 'gs://orbit-wars-shipping/cross_entity'

# Actor cache: prefer the chunked T=10 pair cache if the manifest exists.
PAIR_CACHE_PREFIX = 'pair_cache_t10'
PAIR_CACHE_OBJECT = f'{PAIR_CACHE_PREFIX}.pt'

# Critic cache. critic30k is the existing labelled cache with player_valid and delta columns.
DATASET = 'critic30k'
CROSS_CACHE_BY_DATASET = {
    'critic30k': 'cross_entity_cache_critic30k.pt',
    'tiny':      'cross_entity_cache_tiny.pt',
    '100k':      'cross_entity_cache_100k.pt',
    'full':      'cross_entity_cache_full.pt',
}
CROSS_CACHE_OBJECT = CROSS_CACHE_BY_DATASET[DATASET]

# Warm-start for actor PairHead. This is the same May baseline used by the previous T=10 PairHead notebooks.
BASELINE_RUN = 'top4_pair2head_film_d256_h8_lr5e-05_b256_30ep_20260521-003000'
BASELINE_OBJECT = f'{ENTITY_BUCKET}/runs/{BASELINE_RUN}/entity_encoder_best.pt'

D_MODEL              = 256
D_PAIR               = 256
ENTITY_N_HEADS       = 8
CROSS_N_HEADS        = 8
CROSS_N_LAYERS       = 2
DUAL_N_HEADS         = 8
CONDITIONER_N_LAYERS = 3
HEAD_N_LAYERS        = 3
MAX_PLANETS          = 64
MAX_FLEETS           = 1024
SEED                 = 1729

PAIR_BATCH_SIZE      = 128
PAIR_EPOCHS          = 20
PAIR_LR              = 1e-4
PAIR_WEIGHT_DECAY    = 1e-4
PAIR_POS_WEIGHT      = 600.0
PAIR_NUM_WORKERS     = 2
PAIR_VAL_FRAC        = 0.10
PAIR_TEST_FRAC       = 0.10

CRITIC_BATCH_SIZE    = 256
CRITIC_EPOCHS        = 10
CRITIC_LR            = 5e-4
CRITIC_WEIGHT_DECAY  = 1e-4
CRITIC_NUM_WORKERS   = 2
CRITIC_NUM_LOAD_WORKERS = 8
CRITIC_BACKBONE_LR_MULT = 1.0  # consolidator starts fresh in the actor ckpt; train it at full critic LR.

# Leave false for normal use. If the selected cross cache is stale/missing labels,
# set true after staging CSV shards and rerun the optional rebuild cell below.
REBUILD_CROSS_CACHE_IF_STALE = False
REBUILD_PER_SPLIT_CAP = None

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'dataset={DATASET} cross_cache={CROSS_CACHE_OBJECT} device={DEVICE}')


## 1. Authenticate

In [ ]:
from google.colab import auth
auth.authenticate_user()
print('authenticated')


## 2. Pull code, weights, caches, and baseline

In [ ]:
import concurrent.futures
import hashlib
import json
import os
import shutil
import subprocess
import time
from pathlib import Path

WORK = Path('/content/orbit-wars')
WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)

PAIR_CACHE = WORK / 'pair_cache.pt'
CROSS_CACHE_PATH = WORK / 'data/datasets/cross_entity' / CROSS_CACHE_OBJECT
CROSS_MANIFEST_PATH = WORK / 'data/datasets/cross_entity/manifest.json'
BASELINE_CKPT = WORK / 'baseline_entity_encoder_best.pt'

for rel in ('agents', 'scripts', 'ckpts', 'data/runs'):
    shutil.rmtree(WORK / rel, ignore_errors=True)
for rel in ('data/datasets/cross_entity', 'data/datasets/entity', 'data/datasets/fleet', 'data/datasets/planet'):
    (WORK / rel).mkdir(parents=True, exist_ok=True)

def run(cmd, **kwargs):
    return subprocess.run(cmd, check=True, text=True, **kwargs)

def gcs_size(url: str) -> int | None:
    try:
        out = subprocess.run(
            ['gcloud', 'storage', 'objects', 'describe', url, '--format=value(size)'],
            check=True, capture_output=True, text=True,
        )
    except subprocess.CalledProcessError:
        return None
    s = out.stdout.strip()
    return int(s) if s else None

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as fh:
        for block in iter(lambda: fh.read(8 << 20), b''):
            h.update(block)
    return h.hexdigest()

def cp_if_needed(src: str, dst: Path, *, force: bool = False):
    dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    remote_size = gcs_size(src)
    if remote_size is None:
        raise FileNotFoundError(src)
    if dst.exists() and not force and dst.stat().st_size == remote_size:
        print(f'  cached {dst.name} ({remote_size/1024**3:.2f} GB)')
        return dst.name, 0.0, remote_size
    if dst.exists():
        dst.unlink()
    t0 = time.time()
    print(f'  pulling {src} -> {dst.name} ...', flush=True)
    run(['gcloud', 'storage', 'cp', src, str(dst)])
    return dst.name, time.time() - t0, dst.stat().st_size

def pull_chunked_or_single(bucket: str, object_name: str, dst: Path, *, manifest_names: list[str] | None = None):
    dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    manifest_names = manifest_names or [f'{object_name}.manifest.json', f'{Path(object_name).stem}.manifest.json']
    manifest_local = WORK / f'{Path(object_name).name}.manifest.json'
    manifest = None
    manifest_src = None
    for name in manifest_names:
        src = f'{bucket}/{name}'
        if gcs_size(src) is None:
            continue
        cp_if_needed(src, manifest_local, force=True)
        manifest = json.loads(manifest_local.read_text())
        manifest_src = src
        break
    if manifest is None:
        return cp_if_needed(f'{bucket}/{object_name}', dst)

    chunks = manifest.get('chunks') or manifest.get('parts') or []
    total_bytes = int(manifest.get('total_bytes') or manifest.get('bytes') or 0)
    if dst.exists() and total_bytes and dst.stat().st_size == total_bytes:
        print(f'  cached assembled {dst.name} from {manifest_src}')
        return dst.name, 0.0, dst.stat().st_size
    if not chunks:
        return cp_if_needed(f'{bucket}/{object_name}', dst)

    part_dir = WORK / f'{Path(object_name).stem}_parts'
    part_dir.mkdir(parents=True, exist_ok=True)
    max_workers = max(1, min(16, len(chunks)))
    print(f'  pulling {len(chunks)} chunks for {object_name} with {max_workers} workers ...', flush=True)

    def pull_part(spec):
        name = spec['name']
        src = name if str(name).startswith('gs://') else f'{bucket}/{name}'
        part = part_dir / Path(name).name
        expected = int(spec.get('size_bytes') or spec.get('bytes') or spec.get('size') or 0)
        expected_sha = spec.get('sha256')
        if part.exists() and (not expected or part.stat().st_size == expected):
            if expected_sha is None or sha256_file(part) == expected_sha:
                return part
        if part.exists():
            part.unlink()
        run(['gcloud', 'storage', 'cp', src, str(part)])
        if expected and part.stat().st_size != expected:
            raise RuntimeError(f'chunk size mismatch for {name}: {part.stat().st_size} != {expected}')
        if expected_sha and sha256_file(part) != expected_sha:
            raise RuntimeError(f'chunk sha mismatch for {name}')
        return part

    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as pool:
        list(pool.map(pull_part, chunks))

    if dst.exists():
        dst.unlink()
    print(f'  assembling -> {dst.name} ...', flush=True)
    with dst.open('wb') as out:
        for spec in chunks:
            part = part_dir / Path(spec['name']).name
            with part.open('rb') as fh:
                for block in iter(lambda: fh.read(8 << 20), b''):
                    out.write(block)
    if total_bytes and dst.stat().st_size != total_bytes:
        raise RuntimeError(f'assembled size mismatch for {dst}: {dst.stat().st_size} != {total_bytes}')
    return dst.name, 0.0, dst.stat().st_size

# Pull independent artifacts in parallel. The cache helpers skip local copies when size already matches.
tasks = []
with concurrent.futures.ThreadPoolExecutor(max_workers=6) as pool:
    tasks.append(pool.submit(cp_if_needed, f'{CROSS_BUCKET}/code.tgz', WORK / 'code.tgz'))
    tasks.append(pool.submit(cp_if_needed, f'{CROSS_BUCKET}/weights.tgz', WORK / 'weights.tgz'))
    tasks.append(pool.submit(cp_if_needed, BASELINE_OBJECT, BASELINE_CKPT))
    tasks.append(pool.submit(cp_if_needed, f'{CROSS_BUCKET}/manifest.json', CROSS_MANIFEST_PATH))
    tasks.append(pool.submit(
        pull_chunked_or_single,
        ENTITY_BUCKET,
        PAIR_CACHE_OBJECT,
        PAIR_CACHE,
        manifest_names=[f'{PAIR_CACHE_PREFIX}.manifest.json', f'{PAIR_CACHE_OBJECT}.manifest.json'],
    ))
    tasks.append(pool.submit(pull_chunked_or_single, CROSS_BUCKET, CROSS_CACHE_OBJECT, CROSS_CACHE_PATH))
    for fut in concurrent.futures.as_completed(tasks):
        name, dt, size = fut.result()
        print(f'  ready {name}: {size/1024**3:.2f} GB in {dt:.1f}s')

print('pull complete')


## 3. Extract code and verify model/cache schema

In [ ]:
# Keep the large caches; only refresh extracted Python and ckpts.
!rm -rf agents scripts ckpts
!tar xzf code.tgz
!tar xzf weights.tgz

import gc
import importlib
import sys
for name in list(sys.modules):
    if name.startswith('agents') or name.startswith('scripts'):
        del sys.modules[name]
importlib.invalidate_caches()
gc.collect()
!find . -type d -name __pycache__ -exec rm -rf {} + 2>/dev/null || true
!ls -lah code.tgz weights.tgz pair_cache.pt data/datasets/cross_entity/{CROSS_CACHE_OBJECT} data/datasets/cross_entity/manifest.json


In [ ]:
from pathlib import Path
import torch
import agents
from agents.transformer_v2.history import HISTORY_OFFSETS, N_HISTORY
from agents.transformer_v2.pretrain.entity_encoder import EntityPretrainModel
from agents.transformer_v2.pretrain.cross_entity import (
    CROSS_ENTITY_VALUE_HORIZONS,
    CachedCrossEntitySnapshotDataset,
    CrossEntityCriticModel,
)

print(f'agents: {agents.__file__}')
print(f'HISTORY_OFFSETS: {HISTORY_OFFSETS}')
assert HISTORY_OFFSETS == (45, 40, 35, 30, 25, 20, 15, 10, 5, 0)
assert N_HISTORY == 10

actor_probe = EntityPretrainModel(
    d_model=D_MODEL, n_steps=N_HISTORY, d_pair=D_PAIR,
    entity_n_heads=ENTITY_N_HEADS, cross_n_heads=CROSS_N_HEADS,
    cross_n_layers=CROSS_N_LAYERS, dual_n_heads=DUAL_N_HEADS,
    conditioner_n_layers=CONDITIONER_N_LAYERS, head_n_layers=HEAD_N_LAYERS,
    with_consolidator=True, skip_l34=False,
)
assert actor_probe.consolidator is not None
assert actor_probe.dual_role is not None and actor_probe.joint_role is not None
print('actor probe OK: consolidator + L3/L4 present')

critic_probe = CrossEntityCriticModel(d_model=D_MODEL)
print(f'critic probe OK: params={sum(p.numel() for p in critic_probe.parameters()):,}')
assert hasattr(critic_probe, 'pair_compare')
assert hasattr(critic_probe, 'is_ahead_head')

ds = CachedCrossEntitySnapshotDataset(str(CROSS_CACHE_PATH))
print(f'cross cache rows: {len(ds):,}')
sample = ds[0]
required = ['player_valid']
for h in CROSS_ENTITY_VALUE_HORIZONS:
    required += [f'leader_seat_t_plus_{h}', f'valid_global_t_plus_{h}']
missing = [k for k in required if k not in sample]
if missing:
    raise RuntimeError(
        f'cross cache is missing {missing}. Use the existing data builder '
        f'cell with REBUILD_CROSS_CACHE_IF_STALE=True only if no labelled cache exists.'
    )
assert tuple(sample['player_valid'].shape) == (4,)
assert float(sample['player_valid'].sum()) > 0.0
print('critic labels available:', {k: sample[k].tolist() if hasattr(sample[k], 'tolist') else sample[k] for k in required[:5]})
del ds, sample, actor_probe, critic_probe
gc.collect()


## 3b. Optional: rebuild cross cache only if stale

Normal path: skip this cell. It exists so the notebook can use the existing builder if a future selected cache is stale or missing labels. It does not upload or duplicate a cache unless you explicitly enable it.


In [ ]:
if REBUILD_CROSS_CACHE_IF_STALE:
    import concurrent.futures
    import json
    import shutil
    import subprocess
    from pathlib import Path

    shard_manifest = WORK / 'cross_entity_dataset_shard.manifest.json'
    cp_if_needed(f'{CROSS_BUCKET}/cross_entity_dataset_shard.manifest.json', shard_manifest, force=True)
    manifest = json.loads(shard_manifest.read_text())
    objects = manifest.get('objects', [])
    shard_dir = WORK / 'cross_dataset_shards'
    shard_dir.mkdir(parents=True, exist_ok=True)

    def pull_and_extract(obj):
        name = obj['name']
        local = shard_dir / name
        cp_if_needed(f'{CROSS_BUCKET}/{name}', local)
        subprocess.run(['tar', '-C', str(WORK / 'data/datasets'), '-xzf', str(local)], check=True)
        return name

    with concurrent.futures.ThreadPoolExecutor(max_workers=min(8, len(objects))) as pool:
        for name in pool.map(pull_and_extract, objects):
            print('  extracted', name)

    cmd = [
        'python', '-u', '-m', 'scripts.build_cross_entity_cache',
        '--out', str(CROSS_CACHE_PATH),
        '--max-planets', str(MAX_PLANETS),
        '--max-fleets', str(MAX_FLEETS),
    ]
    if REBUILD_PER_SPLIT_CAP is not None:
        cmd += ['--per-split-cap', str(REBUILD_PER_SPLIT_CAP)]
    subprocess.run(cmd, check=True)
    print(f'rebuilt {CROSS_CACHE_PATH} ({CROSS_CACHE_PATH.stat().st_size/1024**3:.2f} GB)')
else:
    print('skip rebuild; using existing labelled cross cache')


## 4. Stage run directories

In [ ]:
import shutil
from pathlib import Path

PLANET_RUN_DIR = WORK / 'ckpts/planet'
FLEET_RUN_DIR  = WORK / 'ckpts/fleet'
COMET_RUN_DIR  = WORK / 'ckpts/comet'
for d in (PLANET_RUN_DIR, FLEET_RUN_DIR, COMET_RUN_DIR):
    d.mkdir(parents=True, exist_ok=True)
shutil.copy(WORK / 'planet_encoder_best.pt', PLANET_RUN_DIR / 'planet_encoder_best.pt')
shutil.copy(WORK / 'fleet_encoder_best.pt',  FLEET_RUN_DIR  / 'fleet_encoder_best.pt')
shutil.copy(WORK / 'comet_past_best.pt',     COMET_RUN_DIR  / 'comet_past_best.pt')

BASELINE_DIR = WORK / f'ckpts/baseline/{BASELINE_RUN}'
BASELINE_DIR.mkdir(parents=True, exist_ok=True)
INIT_FROM_ENTITY = BASELINE_DIR / 'entity_encoder_best.pt'
shutil.copy(BASELINE_CKPT, INIT_FROM_ENTITY)

for tag, p in (
    ('planet', PLANET_RUN_DIR / 'planet_encoder_best.pt'),
    ('fleet',  FLEET_RUN_DIR  / 'fleet_encoder_best.pt'),
    ('comet',  COMET_RUN_DIR  / 'comet_past_best.pt'),
    ('actor_init', INIT_FROM_ENTITY),
):
    c = torch.load(p, map_location='cpu', weights_only=False)
    cfg = c.get('config', {})
    print(f'{tag:10s}: d_model={cfg.get("d_model")} epoch={c.get("epoch")} path={p}')


## 5. Stage 1: train PairHead actor

In [ ]:
import time
TS = time.strftime('%Y%m%d-%H%M%S')
PAIR_RUN_TAG = f'pair_T10_L3L4_withcons_d{D_MODEL}_b{PAIR_BATCH_SIZE}_{PAIR_EPOCHS}ep_lr{PAIR_LR:g}_{TS}'
PAIR_OUT_DIR = WORK / f'data/runs/entity/{PAIR_RUN_TAG}'
PAIR_CACHE_PATH = PAIR_CACHE
print('pair run:', PAIR_RUN_TAG)
print('pair out:', PAIR_OUT_DIR)


In [ ]:
!python -u -m agents.transformer_v2.pretrain.entity_encoder   --planet-run-dir $PLANET_RUN_DIR   --fleet-run-dir  $FLEET_RUN_DIR   --comet-run-dir  $COMET_RUN_DIR   --pair-cache-path $PAIR_CACHE_PATH   --out-dir $PAIR_OUT_DIR   --d-model $D_MODEL   --d-pair $D_PAIR   --entity-n-heads $ENTITY_N_HEADS   --cross-n-heads $CROSS_N_HEADS   --cross-n-layers $CROSS_N_LAYERS   --dual-n-heads $DUAL_N_HEADS   --conditioner-n-layers $CONDITIONER_N_LAYERS   --head-n-layers $HEAD_N_LAYERS   --init-from-entity-ckpt $INIT_FROM_ENTITY   --freeze-perception   --batch-size $PAIR_BATCH_SIZE   --epochs $PAIR_EPOCHS   --lr $PAIR_LR   --weight-decay $PAIR_WEIGHT_DECAY   --max-planets $MAX_PLANETS   --max-fleets $MAX_FLEETS   --pair-pos-weight $PAIR_POS_WEIGHT   --val-frac $PAIR_VAL_FRAC   --test-frac $PAIR_TEST_FRAC   --num-workers $PAIR_NUM_WORKERS   --seed $SEED   --device $DEVICE


## 6. Stage 2: train redesigned critic

In [ ]:
ACTOR_BEST = PAIR_OUT_DIR / 'entity_encoder_best.pt'
assert ACTOR_BEST.exists(), ACTOR_BEST
CRITIC_RUN_TAG = f'paircritic_T10_{DATASET}_d{D_MODEL}_b{CRITIC_BATCH_SIZE}_{CRITIC_EPOCHS}ep_lr{CRITIC_LR:g}_{TS}'
CRITIC_OUT_DIR = WORK / f'data/runs/cross_entity/{CRITIC_RUN_TAG}'
print('critic run:', CRITIC_RUN_TAG)
print('critic init:', ACTOR_BEST)
print('critic out:', CRITIC_OUT_DIR)


In [ ]:
!python -u -m agents.transformer_v2.pretrain.cross_entity   --train-mode frozen   --fleet-run-dir  $FLEET_RUN_DIR   --planet-run-dir $PLANET_RUN_DIR   --entity-run-dir $PAIR_OUT_DIR   --out-dir $CRITIC_OUT_DIR   --cross-cache-path $CROSS_CACHE_PATH   --init-from $ACTOR_BEST   --unfreeze consolidator   --backbone-lr-mult $CRITIC_BACKBONE_LR_MULT   --aux-posterior   --lambda-cons 0.2   --lambda-post 0.5   --d-model $D_MODEL   --batch-size $CRITIC_BATCH_SIZE   --epochs $CRITIC_EPOCHS   --lr $CRITIC_LR   --weight-decay $CRITIC_WEIGHT_DECAY   --head-set critic   --seed $SEED   --num-load-workers $CRITIC_NUM_LOAD_WORKERS   --num-workers $CRITIC_NUM_WORKERS   --device $DEVICE


## 7. Merge trained consolidator into actor checkpoint

In [ ]:
import json
import torch
from pathlib import Path

CRITIC_BEST = CRITIC_OUT_DIR / 'cross_entity_best.pt'
MERGED_ACTOR = PAIR_OUT_DIR / 'entity_encoder_best_with_critic_consolidator.pt'
assert ACTOR_BEST.exists(), ACTOR_BEST
assert CRITIC_BEST.exists(), CRITIC_BEST

actor_ckpt = torch.load(ACTOR_BEST, map_location='cpu', weights_only=False)
critic_ckpt = torch.load(CRITIC_BEST, map_location='cpu', weights_only=False)
actor_state = dict(actor_ckpt['model'])
critic_state = critic_ckpt['model']

copied = []
for key, value in critic_state.items():
    if key.startswith('consolidator.'):
        if key not in actor_state:
            raise KeyError(f'actor checkpoint has no {key}; actor must be trained with with_consolidator=True')
        actor_state[key] = value.detach().cpu()
        copied.append(key)
if not copied:
    raise RuntimeError('critic checkpoint had no consolidator.* keys')

merged = dict(actor_ckpt)
merged['model'] = actor_state
merged_cfg = dict(actor_ckpt.get('config', {}))
merged_cfg['with_consolidator'] = True
merged_cfg['critic_consolidator_from'] = str(CRITIC_BEST)
merged_cfg['critic_paircompare_from'] = str(CRITIC_BEST)
merged['config'] = merged_cfg
merged['merged_critic_consolidator_keys'] = copied
torch.save(merged, MERGED_ACTOR)
print(f'wrote {MERGED_ACTOR} ({MERGED_ACTOR.stat().st_size/1024**2:.1f} MB); copied {len(copied)} consolidator tensors')

# Quick load sanity: actor ckpt has consolidator; critic ckpt has pair_compare.
from agents.transformer_v2.pretrain.entity_encoder import EntityPretrainModel
from agents.transformer_v2.pretrain.cross_entity import CrossEntityCriticModel
m = EntityPretrainModel(
    d_model=D_MODEL, n_steps=N_HISTORY, d_pair=D_PAIR,
    entity_n_heads=ENTITY_N_HEADS, cross_n_heads=CROSS_N_HEADS,
    cross_n_layers=CROSS_N_LAYERS, dual_n_heads=DUAL_N_HEADS,
    conditioner_n_layers=CONDITIONER_N_LAYERS, head_n_layers=HEAD_N_LAYERS,
    with_consolidator=True, skip_l34=False,
)
res = m.load_state_dict(merged['model'], strict=False)
print(f'merged actor load: missing={len(res.missing_keys)} unexpected={len(res.unexpected_keys)}')
c = CrossEntityCriticModel(d_model=D_MODEL)
res2 = c.load_state_dict(critic_ckpt['model'], strict=False)
print(f'critic load: missing={len(res2.missing_keys)} unexpected={len(res2.unexpected_keys)}')


## 8. Upload outputs

In [ ]:
import subprocess
from pathlib import Path

assert PAIR_OUT_DIR.is_dir(), PAIR_OUT_DIR
assert CRITIC_OUT_DIR.is_dir(), CRITIC_OUT_DIR

subprocess.run(['gcloud', 'storage', 'cp', '--recursive', str(PAIR_OUT_DIR), f'{ENTITY_BUCKET}/runs/'], check=True)
subprocess.run(['gcloud', 'storage', 'cp', '--recursive', str(CRITIC_OUT_DIR), f'{CROSS_BUCKET}/runs/'], check=True)
print(f'uploaded actor:  {ENTITY_BUCKET}/runs/{PAIR_OUT_DIR.name}/')
print(f'uploaded critic: {CROSS_BUCKET}/runs/{CRITIC_OUT_DIR.name}/')
subprocess.run(['gcloud', 'storage', 'ls', '--long', '--readable-sizes', f'{ENTITY_BUCKET}/runs/{PAIR_OUT_DIR.name}/'], check=False)
subprocess.run(['gcloud', 'storage', 'ls', '--long', '--readable-sizes', f'{CROSS_BUCKET}/runs/{CRITIC_OUT_DIR.name}/'], check=False)
